# Optimizing modeling code with Popcorn

[`modeling_butter.py`](./modeling_butter.py) defines **Butter**, a quirky hybrid language model that alternates [Gated DeltaNet](https://arxiv.org/abs/2412.06464) and attention blocks. It is written on popcorn kernels from the start: RMSNorm, RoPE, SwiGLU, the delta-rule scan, and the linear-cross-entropy head all go through the dispatcher, which routes each call to the fastest validated backend for this GPU. Attention stays on `F.scaled_dot_product_attention` — it is already optimal, and popcorn does not need to replace what torch does well.

Every kernel also carries its ground-truth **reference**, a plain PyTorch implementation. `ButterConfig(dispatch=False)` binds the same call sites to those references instead — that is our unoptimized baseline: same code, same weights, no dispatch.

In [1]:
import torch

from modeling_butter import ButterConfig, ButterForCausalLM

torch.manual_seed(0)
device, dtype = "cuda", torch.bfloat16

config = ButterConfig()
model = ButterForCausalLM(config).to(device, dtype)  # tuned popcorn dispatch
baseline = ButterForCausalLM(ButterConfig(dispatch=False)).to(device, dtype)  # plain-torch references
baseline.load_state_dict(model.state_dict(), assign=True)  # same weights, shared storage

input_ids = torch.randint(config.vocab_size, (4, 1024), device=device)
labels = torch.randint(config.vocab_size, (4, 1024), device=device)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters")

340M parameters


## Forcing a backend

Dispatch is automatic, but every kernel can be pinned — per call, per region, or (as the config does) structurally via `op.reference`.

In [2]:
from popcorn import kernels

x = torch.randn(4, 1024, config.hidden_size, device=device, dtype=dtype)
weight = torch.ones(config.hidden_size, device=device, dtype=dtype)

y = kernels.rms_norm(x, weight)  # tuned dispatch picks the recorded winner
y_torch = kernels.rms_norm(x, weight, backend="torch")  # force one call
y_liger = kernels.rms_norm["liger"](x, weight)  # same thing, subscript style
with kernels.rms_norm["torch"]:  # scope a whole region
    y_scoped = kernels.rms_norm(x, weight)

print(kernels.rms_norm, "", sep="\n")
kernels.linear_cross_entropy  # note the gates: every fused loss backend is float32-only

/home/timor/popcorn/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


rms_norm(x: jaxtyping.Float[Tensor, '... normalized_shape'], weight: jaxtyping.Float[Tensor, 'normalized_shape'], bias: jaxtyping.Float[Tensor, 'normalized_shape'] | None = None, eps: float = 1e-06) -> jaxtyping.Float[Tensor, '... normalized_shape']
  fla
  liger  gates={'bias': typing.Literal[None]}
  quack
  unsloth  fwd-only  gates={'bias': typing.Literal[None]}
  torch



linear_cross_entropy(x: jaxtyping.Float[Tensor, 'tokens hidden'], weight: jaxtyping.Float[Tensor, 'vocab hidden'], labels: jaxtyping.Int[Tensor, 'tokens'], bias: jaxtyping.Float[Tensor, 'vocab'] | None = None, ignore_index: int = -100, label_smoothing: float = 0.0, reduction: Literal['mean', 'sum'] = 'mean') -> jaxtyping.Float[Tensor, '']
  fla  gates={'x': <class 'jaxtyping.Float32[Tensor, 'tokens hidden']'>}
  liger  gates={'x': <class 'jaxtyping.Float32[Tensor, 'tokens hidden']'>}
  quack  fwd-only  gates={'x': <class 'jaxtyping.Float32[Tensor, 'tokens hidden']'>, 'bias': typing.Literal[None], 'label_smoothing': typing.Literal[0.0]}
  torch

## Profile the whole model with popcorn's harness

`popcorn.bench.compare` is the same machinery the kernel test suite runs on every backend: it times both sides with triton's `do_bench` (L2 cache cleared between runs, median of many reps), tracks peak memory, and grades the outputs against a float64 run of the reference. Here we hand it the whole model — tuned dispatch as `mine`, the plain-torch baseline as `reference`.

First inference.

In [3]:
from popcorn.bench import compare

batch = {"input_ids": input_ids, "labels": labels}


def report(result, what):
    bench, gauge = result.bench, result.fwd["out0"]
    speed = bench["ref_fwd_ms"] / bench["fwd_ms"]
    print(f"{what:10s} torch {bench['ref_fwd_ms']:8.1f} ms -> popcorn {bench['fwd_ms']:6.1f} ms   ({speed:.1f}x faster)")
    if "fwd_mem_mb" in bench:
        print(f"{'':10s} peak  {bench['ref_fwd_mem_mb'] / 1024:8.2f} GB -> {bench['fwd_mem_mb'] / 1024:9.2f} GB")
    print(f"{'':10s} loss deviation {gauge.err:.2e} on scale {gauge.scale:.1f} (reference budget {gauge.budget:.2e})")
    assert gauge.err / gauge.scale < 0.02


def tuned_forward(**batch):
    with torch.no_grad():
        return model(**batch)


def reference_forward(**batch):
    with torch.no_grad():
        return baseline(**batch)


report(compare(tuned_forward, reference_forward, batch, backward=False, repeats=5), "forward")

forward    torch    225.3 ms -> popcorn    6.1 ms   (37.2x faster)
           peak      0.77 GB ->      0.77 GB
           loss deviation 0.00e+00 on scale 10.8 (reference budget 0.00e+00)


Then a full training step — forward, backward, and gradient reset inside the timed callable.

In [4]:
def tuned_step(**batch):
    loss = model(**batch)
    loss.backward()
    model.zero_grad(set_to_none=True)
    return loss.detach()


def reference_step(**batch):
    loss = baseline(**batch)
    loss.backward()
    baseline.zero_grad(set_to_none=True)
    return loss.detach()


report(compare(tuned_step, reference_step, batch, backward=False, repeats=3), "train step")

train step torch   1238.5 ms -> popcorn   15.5 ms   (80.2x faster)
           peak     18.89 GB ->      2.22 GB
           loss deviation 0.00e+00 on scale 10.8 (reference budget 0.00e+00)


## Where the difference comes from

The delta-rule scan is nearly all of it, on both axes: a python loop launches a handful of kernels per token where fla's chunked kernel processes the sequence in a few launches, and under training that loop keeps a thousand steps of intermediates alive for backward, which is where the baseline's peak memory goes. The norms, rope, and swiglu ride along on fused liger kernels.

One op deliberately stays put: the loss head. In bfloat16 every fused linear-cross-entropy backend is gated out — they return the loss in float32, failing popcorn's dtype round-trip contract (the `Float32` gates in the table above) — so dispatch keeps the torch reference there. Run the model in float32 and the fused backends take over.

Every backend choice above came from benchmark data recorded for this GPU, and the first call per kernel validated the winner against its reference before trusting it. Browse the measured landscape in the [kernel explorer](https://tilde-research.github.io/popcorn/kernels/), or force any backend as shown above. To inject these kernels into a model you cannot edit, see the experimental `torch.compile` integration in the next notebook.